# AWQ calibration data debug notebook

このノートブックは `src/llmlab/utils/awq.py::prepare_awq_calib_data` および AutoAWQ の挙動を単体で確認するためのデバッグ用ツールです。
大規模モデルを使う前に、キャリブレーションサンプルの前処理が想定どおり動作するか確認できます。


In [ ]:
# --- Environment setup (Google Colab) ---
import os
import sys
from typing import List

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover
    get_ipython = None  # type: ignore

def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython environment is required (e.g. Google Colab).")
    return ip

def clone_repo(url: str, target: str, branch: str | None = None) -> None:
    if os.path.exists(target):
        print("Reusing existing repository:", target)
        return
    ip = _require_ipython()
    if branch:
        print(f"Cloning repository: {url} (branch={branch})")
        ip.system(f"git clone --branch {branch} --single-branch {url} {target}")
    else:
        print("Cloning repository:", url)
        ip.system(f"git clone {url} {target}")

def pip_install(packages: List[str]) -> None:
    packages = list(packages)
    if not packages:
        return
    ip = _require_ipython()
    quoted = " ".join(f{pkg} for pkg in packages)
    print("pip install:", packages)
    ip.run_line_magic("pip", f"install --upgrade {quoted}")

REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR
REPO_BRANCH = "main"

BASE_PACKAGES = [
    "transformers>=4.56.0,<4.57.0",
    "peft>=0.17.0,<0.18.0",
    "autoawq",
    "accelerate",
    "datasets",
    "safetensors",
]
ADDITIONAL_PACKAGES: List[str] = []

RUN_SETUP = False  # set to True if you need to clone/install
if RUN_SETUP:
    clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
    pip_install(BASE_PACKAGES + ADDITIONAL_PACKAGES)


In [ ]:
# --- Configure calibration samples and model ---
from pathlib import Path

REPO_ROOT = Path(REPO_DIR).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CALIBRATION_SAMPLES = [
    "You are a helpful assistant. Please follow the instructions strictly.",
    "Summarise the following enterprise deck in roughly 200 Japanese characters.",
]

# Use a small model by default for quick tests. Change to your target model as needed.
TEST_TOKENIZER_NAME = "facebook/opt-125m"
TEST_AWQ_MODEL_NAME = TEST_TOKENIZER_NAME

print(f"Repository root: {REPO_ROOT}")
print(f"Calibration samples: {len(CALIBRATION_SAMPLES)} item(s)")


In [ ]:
# --- Convert samples and optionally run AWQ quantisation ---
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM
from src.llmlab.utils.awq import prepare_awq_calib_data

print("Loading tokenizer:", TEST_TOKENIZER_NAME)
tokenizer = AutoTokenizer.from_pretrained(TEST_TOKENIZER_NAME, trust_remote_code=True)
calib_data, calib_text_column = prepare_awq_calib_data(CALIBRATION_SAMPLES)

print(f"Prepared calibration type: {type(calib_data).__name__}")
print(f"Text column hint: {calib_text_column}")

calib_payload = calib_data
if isinstance(calib_data, list) and calib_data:
    if isinstance(calib_data[0], str):
        encoded = []
        for idx, sample in enumerate(calib_data):
            tokens = tokenizer.encode(sample, add_special_tokens=False)
            print(f"Sample {idx}: {len(tokens)} token(s)")
            if tokens:
                encoded.append(tokens)
        if not encoded:
            raise ValueError("Calibration samples produced no tokens.")
        calib_payload = encoded
    else:
        calib_payload = calib_data

RUN_QUANT_TEST = False  # set to True to run AutoAWQ quantisation
if RUN_QUANT_TEST:
    print("Loading AWQ model:", TEST_AWQ_MODEL_NAME)
    awq_model = AutoAWQForCausalLM.from_pretrained(
        TEST_AWQ_MODEL_NAME,
        trust_remote_code=True,
        device_map={"": "cpu"},
    )
    quant_kwargs = dict(
        tokenizer=tokenizer,
        quant_config={"w_bit": 4, "q_group_size": 128},
        calib_data=calib_payload,
    )
    if calib_text_column:
        quant_kwargs["text_column"] = calib_text_column
    try:
        awq_model.quantize(**quant_kwargs)
    except Exception as exc:
        print("Quantisation raised:", type(exc).__name__, exc)
        raise
    else:
        print("Quantisation completed successfully.")
